# Concepts

This page introduces the core ideas behind SysSimX. It builds a mental model in layers: first *why* co-simulation is needed, then the building blocks (components, ports, connections), and finally how SysSimX orchestrates them in time (execution order, algebraic loops, master algorithms, and events).

If you just want to run your first simulation, start with the {doc}`Quickstart <02_quickstart>` and come back here when you want to understand what happens under the hood.

## Why Co-Simulation?

Realistic engineering systems are *heterogeneous*: a controller may be designed in Modelica and exported as an FMU, a structure may be modeled with finite elements, and human motion may come from a musculoskeletal model. No single tool covers all of these domains well.

**Co-simulation** couples such subsystems while each keeps its own solver. Every subsystem advances itself internally and only exchanges input/output signals with the others at discrete points in time:

<img src="../_static/concepts/coupled_subsystems.svg" alt="Two coupled subsystems exchanging inputs and outputs" width="520"/>

Data is exchanged on a **communication grid**: the *macro step* $H_k$ is the interval between two communication points $T_k$ and $T_{k+1}$, while each subsystem may internally take several smaller *micro steps* $\Delta t$ with its own solver:

<img src="../_static/concepts/communication_grid.svg" alt="Macro steps between communication points and internal micro steps" width="620"/>

Between communication points, each subsystem extrapolates (holds) its inputs. This decoupling is what makes it possible to combine tools — but it also introduces the coordination questions (ordering, loops, events) that the rest of this page is about.

## The Big Picture

SysSimX is a Python framework that acts as the **master** of such a co-simulation: it owns the components, resolves their coupling, and advances them consistently in time. The framework is organized in four layers between your user code and the external simulation tools:

<img src="../_static/concepts/syssimx_architecture.svg" alt="SysSimX layered architecture" width="680"/>

- **Adapter layer** — wraps external tools (FMUs, OpenSim, NGSolve) behind one uniform component interface. New tools are integrated by writing a new adapter.
- **Core abstractions** — `CoSimComponent`, `MultiComponent`, `Connection`, and `EventConnection`: the vocabulary you use to describe a system.
- **System orchestration** — the `System` container analyzes the coupling structure (dependency graph, execution order, algebraic loops).
- **Execution layer** — the master algorithms (Jacobi, Gauss–Seidel, IJCSA, Hybrid) that actually advance the system in time.

As a user you mostly interact with the top: create components, connect them, call `system.run()`, and read back the `SimulationResult` returned by `system.run()`.

## Mental Model: System, Component, Connection

Four terms are enough to describe any SysSimX model:

| Concept | Meaning |
|---|---|
| **Component** | A model with input ports `u`, output ports `y`, parameters, and (optionally) internal state. |
| **Connection** | A directed signal path from an output port to an input port. |
| **System** | The container that owns components and connections and orchestrates execution. |
| **Step** | One advance of the whole system by a macro step `dt`. |

The figure below shows a small system of four components. Solid arrows are **connections** between ports; red dashed arrows *inside* a component mark **direct feedthrough** (the output depends on the input at the same instant); blue dashed arrows are **event connections** carrying discrete events instead of continuous signals:

<img src="../_static/concepts/system_topology.svg" alt="Example system topology with connections, direct feedthrough, and event connections" width="520"/>

In code, the same structure reads almost like the picture:

```python
system = System("example")
system.add_component(a)
system.add_component(b)
system.add_connection(Connection(
    src_comp=a.name, src_port="y1",
    dst_comp=b.name, dst_port="u1",
))
system.initialize(t0=0.0)
result = system.run(t0=0.0, tf=10.0, dt=0.01)
```

## Components

Every model in SysSimX — whether it wraps an FMU, an NGSolve transient structural-dynamics model, or ten lines of Python — implements the same abstract base class `CoSimComponent`. The base class defines the typed port interface, the lifecycle, and a set of *optional capabilities* (dashed borders below) that the master algorithms can exploit when present:

<img src="../_static/concepts/cosimcomponent_capabilities.svg" alt="CoSimComponent capability map: ports, lifecycle, state management, model structure, hybrid capabilities" width="620"/>

- **Lifecycle** — construction, configuration, initialization, the simulation loop, and reset/release (details below).
- **State management** — components may expose their physical state for inspection or support **rollback**, which the hybrid algorithm uses to localize events precisely.
- **Model structure** — components declare their input–output dependencies (direct feedthrough), which drives the execution-order analysis.
- **Hybrid capabilities** — components may publish event indicators or subscribe to events from others.

Built-in adapters exist for common tools, and custom components are plain Python classes:

| Component | Wraps |
|---|---|
| `FMUComponent` | FMI 2.0 Co-Simulation FMUs |
| `FEMComponent` | NGSolve transient structural dynamics with Newmark integration |
| `OpenSimComponent` | OpenSim musculoskeletal models |
| custom `CoSimComponent` subclass | any Python code |

### The Simulation Step

During the simulation loop, each macro step of a component follows a fixed pattern: the algorithm sets the inputs, calls `do_step(t, dt)`, and reads the outputs. Inside `do_step`, the base class delegates the tool-specific work to two abstract hooks and takes care of time bookkeeping and result recording itself:

<img src="../_static/concepts/component_stepping.svg" alt="Sequence diagram of a simulation step through the CoSimComponent base class" width="640"/>

This split is what makes adapters small: a new component only implements *advance the internal solver* and *write internal state to the output ports* — everything else (port handling, unit conversion, history recording) is inherited.

During a run, `component.get_outputs()` exposes current values. After a run, use the `SimulationResult` returned by `system.run()` for recorded time series, event records, tabular conversion, and export.

## Ports and Units

Ports are the interface of a component. Each port separates an immutable **contract** (`PortSpec`: name, type, direction, optional unit) from its mutable **runtime state** (`PortState`: current value and timestamp):

<img src="../_static/concepts/port_model.svg" alt="Port model: PortSpec contract and PortState runtime value" width="680"/>

Two consequences matter in practice:

1. **Type safety** — a connection is only valid if the two port specs are compatible (matching type, out → in direction). Mistakes are caught at connect time, not as silent numerical errors during the run.
2. **Automatic unit conversion** — units are handled with [Pint](https://pint.readthedocs.io/). If a drive outputs torque in `N*m` and a biomechanics model expects `N*mm`, the connection converts automatically. This is essential when combining tools with different unit conventions.

## Execution Order and Direct Feedthrough

Before the first step, `System.initialize()` performs a **structural analysis** of the coupling. Components and connections form a directed dependency graph; a connection into a **direct-feedthrough** input is a *zero-delay* edge, because the downstream output needs the upstream value at the same instant:

<img src="../_static/concepts/dependency_graph.svg" alt="Dependency graph of components with port-to-port edges" width="380"/>

From this graph SysSimX derives a **generation-based execution order**. Components in the same generation have no zero-delay dependency on each other, so they are candidates for parallel execution; the current algorithms still invoke them serially. Later generations follow the components that feed them. Components whose outputs depend only on internal state break the zero-delay chain:

<img src="../_static/concepts/execution_order.svg" alt="Derivation of the generation-based execution order" width="440"/>

You do not specify this order manually. Assemble the topology first, then call `initialize()` to compute it.


## Algebraic Loops

If zero-delay (direct-feedthrough) edges form a cycle, no valid ordering exists: each component needs the other's output *first*. This is an **algebraic loop**. During structural analysis, SysSimX detects these cycles as strongly connected components (SCCs) of the zero-delay graph:

<img src="../_static/concepts/algebraic_loop_scc.svg" alt="Strongly connected component in the zero-delay graph forming an algebraic loop" width="280"/>

Loops are not an error — they are solved iteratively at each communication point using the **IJCSA** method (Interface Jacobian-based Co-Simulation Algorithm): the loop components are evaluated repeatedly, with a Newton-type update on the interface variables, until the coupling residual converges. Everything outside the loop still runs in the normal generation order.

## Master Algorithms

The master algorithm decides *when* each component steps and *which* input values it sees. SysSimX provides two classic schemes for continuous coupling:

<img src="../_static/concepts/jacobi_gauss_seidel.svg" alt="Jacobi lagged-input versus Gauss-Seidel current-step propagation" width="660"/>

- **Jacobi** - freezes all inputs before advancing component state, so feedthrough signals use values available at the start of the macro step rather than same-step outputs. These advances are mathematically independent, but the current SysSimX implementation invokes them serially.
- **Gauss-Seidel** - steps components sequentially in execution order, so downstream components see current-step values from upstream neighbors.

In addition:

- **IJCSA** - iteratively solves algebraic strongly connected components at a communication point.
- **Hybrid** - adds rollback-based event detection and zero-crossing localization.

Algorithm selection is explicit for continuous systems: a new `System` uses Gauss-Seidel by default, and you can call `set_algorithm(...)` or set `algorithm.type` in a declarative configuration. During `initialize()`, a system with event sources is promoted to `HybridAlgorithm`. Algebraic loops are handled with IJCSA inside the execution path; structural analysis does not otherwise choose an algorithm based on feedthrough chains.

The accuracy difference is visible in a simple feedthrough benchmark. Jacobi shows the characteristic one-step lag at large step sizes, while Gauss-Seidel and IJCSA stay closer to the analytic solution; with decreasing macro step size all schemes converge:

<img src="../_static/concepts/master_algorithm_accuracy.svg" alt="Accuracy comparison of Jacobi, Gauss-Seidel, and IJCSA against an analytic solution for different step sizes" width="660"/>


## Multi-Model Components

Often the *right* model fidelity depends on the operating condition: a rigid-body pendulum is cheap and accurate in free swing, but a finite element model is needed near contact. A `MultiComponent` bundles several models of the *same* physical subsystem behind one unified port interface and switches between them at runtime:

<img src="../_static/concepts/multi_model_component.svg" alt="MultiComponent: unified interface delegating to the active model with mode switching support" width="520"/>

- A **mode selector** decides which model should be active (e.g., based on distance to contact).
- **Hysteresis** prevents rapid back-and-forth switching at the decision boundary.
- A **state adapter** transfers the physical state from the deactivated model to the newly activated one, so the switch is continuous.

From the outside, the rest of the system never notices the switch — the `MultiComponent` looks like any other component.

## Events and Hybrid Simulation

Many systems are **hybrid**: continuous dynamics interrupted by discrete events — a contact, a switch, an impact. At an event time $t_i$, a continuous state may jump ($x^-  \to x^+$, e.g., a velocity reversal at impact) and a discrete mode may change ($q^- \to q^+$):

<img src="../_static/concepts/hybrid_state_event.svg" alt="Hybrid system: continuous state jump and discrete mode change at an event time" width="480"/>

Handling this accurately in a co-simulation requires more than fixed steps. The **hybrid master algorithm** localizes each event by bisection, illustrated below for a pendulum hitting a wall (event indicator $\gamma(t) = \theta(t) - \theta_{\mathrm{wall}}$, ideal elastic impact):

<img src="../_static/concepts/hybrid_event_localization.svg" alt="Event detection and localization by bisection: trial step, bracketing intervals, accepted step to the event time" width="560"/>

1. **Detect** — a trial step $T_k \to T_{k+1}$ shows a sign change of the indicator ($\gamma_k > 0$, $\gamma_{k+1} < 0$): an event lies inside the step. The components are rolled back to their snapshots at $T_k$.
2. **Bisect** — a trial step over the left half $[T_k, T_m]$ shows no crossing ($\gamma_m > 0$), so the event must lie in the right half.
3. **Refine** — stepping continues from $T_m$ with halved intervals ($\Delta t/4$, $\Delta t/8$, …), keeping whichever half brackets the crossing, until the event time $t_{\mathrm{ev}}$ is located within tolerance.
4. **Commit** — the components are restored once more and take one accepted step exactly to $t_{\mathrm{ev}}$; event handlers are dispatched at that dense time (here the impact handler $\omega^+ = -\omega^-$) and subscribed components are notified via **event connections**. Continuous stepping then resumes from $t_{\mathrm{ev}}$.

This is why the **rollback** capability from the component capability map matters: without snapshots, trial steps could not be discarded.

Events can also *cascade*: one component's event changes another component's inputs, which may immediately trigger a follow-up event at the same instant. SysSimX processes such event chains to completion before continuous time resumes:

<img src="../_static/concepts/event_chain.svg" alt="Event chain: a source event propagating through intermediate components to listeners at one instant" width="660"/>

## Putting It All Together

The controlled pendulum below combines everything from this page in one system: a setpoint source, a PID controller, a BLDC drive, a pendulum with wall contact, and a sensor chain (potentiometer → ADC → decoder) closing the feedback loop. The wall contact is a discrete event that reverses the angular velocity and resets the controller's integrator via an event connection:

<img src="../_static/concepts/controlled_pendulum_system.svg" alt="Controlled pendulum case study: setpoint, PID controller, BLDC drive, pendulum with wall contact, and sensor chain" width="820"/>

Reading it with the concepts from this page:

- Each block is a **component** with typed, unit-aware **ports**; the arrows are **connections**.
- The controller and sensor chain have **direct feedthrough**, so the **execution order** matters within each step.
- The closed control loop is broken by the pendulum's internal state — no **algebraic loop** remains.
- The wall contact is a **zero-crossing event**; the dashed red *contact event* line is an **event connection** that resets the PID integrator.
- The pendulum itself can be a **multi-model component**, switching between a cheap rigid-body model in free swing and an FEM model near contact.

This system is developed step by step in the {doc}`controlled pendulum case study <../05_case_study/00_overview>` and its linked notebooks.

## Where to Go Next

- {doc}`Quickstart <02_quickstart>` — build and run your first system in a few lines.
- {doc}`Core tutorials <../03_core_tutorials/01_fundamentals/index>` — build systems from simple components to hybrid simulations.
- {doc}`Controlled pendulum case study <../05_case_study/00_overview>` — follow the complete multi-stage example.
- {doc}`SysSimX package API <../02_api/syssimx>` and {doc}`component API <../02_api/components>` — reference documentation for the framework.